In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import f_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
from matplotlib import pyplot as plt
from sklearn.preprocessing import OneHotEncoder

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [ ]:
# Helper functions
def calculate_metrics(y_test, preds):
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    print(f"Mean Absolute Error (MAE): {mae:.2f}")
    print(f"Mean Squared Error (MSE): {mse:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
    print(f"R-squared (R2) Score: {r2:.2f}")

    return (mae, mse, rmse, r2)
    
def plot_pred_vs_true(y_test, preds):
    MIN_VALUE = 0
    MAX_VALUE = max(y_test)
    plt.figure(figsize=(8, 8))
    plt.scatter(y_test, preds)
    plt.plot([MIN_VALUE, MAX_VALUE], [MIN_VALUE, MAX_VALUE], '--k', label="Correct prediction")

    plt.title("Predicted vs. True Plot")
    plt.xlabel('True Price')
    plt.ylabel('Predicted Price')
    plt.legend()
    plt.tight_layout()

def plot_residual(y_test, preds):
    residuals = y_test - preds
    plt.figure(figsize=(8, 8))
    plt.xlabel('Price')
    plt.ylabel('Residuals')
    plt.title('Residual Plot')
    plt.scatter(preds, residuals)
    plt.axhline(0, linestyle="--")

def z_score_scale(dataset, feature_name):
    feature = list(dataset[feature_name])
    mu = np.mean(feature)
    std = np.std(feature)
    z_score = [round((i - mu) / std, 2) for i in feature]

    dataset[feature_name] = z_score

# Helper functions - Data imputation:
# Source: https://ansumanbhujabal.medium.com/machine-learning-basic-handling-missing-values-in-dataset-ca81914380ed
def fill_mean(dataset, feature_name):
    '''
    The mean is used to impute missing values when 
    dealing with continuous or numeric data, 
    such as age, income, or temperature.
    '''
    feature = list(dataset[feature_name])
    mean = round(np.nanmean(feature),1)
    for i, e in enumerate(feature):
        if pd.isna(e):
            feature[i] = mean

    dataset[feature_name] = feature

def fill_mode(dataset, feature_name):
    '''
    The mode is used for imputation when dealing with categorical data 
    or data with a limited set of distinct categories, 
    like car types, colors, or city names.
    '''
    feature = list(dataset[feature_name])
    frequencies = {}
    for e in set(feature):
        frequencies[e] = feature.count(e)
    max_element = max(frequencies, key=frequencies.get)
    for i, e in enumerate(feature):
        if pd.isna(e):
            feature[i] = max_element
    
    dataset[feature_name] = feature

def fill_median(dataset, feature_name):
    '''
    The median is a good choice for imputation when dealing with skewed data 
    or outliers in continuous or numeric data. 
    It’s less sensitive to extreme values than the mean.
    '''
    feature = list(dataset[feature_name])
    middle_idx = len(feature) // 2 - 1
    ordered_feature_middle_point = sorted(feature)[middle_idx]
    for i, e in enumerate(feature):
        if pd.isna(e):
            feature[i] = ordered_feature_middle_point

    dataset[feature_name] = feature

# Helper functions - Conversion

def category_to_num(dataset, feature_name):
    feature = list(dataset[feature_name])
    elements = set(feature)
    
    categories = {}
    for i, e in enumerate(elements):
        if not categories.get(e):
            categories[e] = float(i)

    for i, e in enumerate(feature):
        feature[i] = categories[e]
    
    dataset[feature_name] = feature

def remove_row_due_missing_element(dataset, feature_name):
    return dataset.dropna(subset=[feature_name])


def detect_outliers_and_clip(dataset, feature_name):
    feature = dataset[feature_name]
    Q1 = feature.quantile(0.25)
    Q3 = feature.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR 
    dataset[feature_name] = feature.clip(lower=lower_bound, upper=upper_bound)

def one_hot_encode_train_test(X_train, X_test, feature_name):
    train_feature = X_train[[feature_name]]
    test_feature = X_test[[feature_name]]

    encoder.fit(train_feature)

    encoded_columns = encoder.get_feature_names_out([feature_name])

    train_encoded = encoder.transform(train_feature)
    test_encoded = encoder.transform(test_feature)

    train_encoded_df = pd.DataFrame(train_encoded, columns=encoded_columns, index=X_train.index)
    test_encoded_df = pd.DataFrame(test_encoded, columns=encoded_columns, index=X_test.index)

    X_train = pd.concat([X_train.drop(columns=[feature_name]), train_encoded_df], axis=1)

    X_test = pd.concat([X_test.drop(columns=[feature_name]), test_encoded_df], axis=1)

    return X_train, X_test


In [ ]:
# Download aws s3 cp "s3://$CURATED_BUCKET/curated/test/sample_input.csv" ./processed_output.csv
dataset = pd.read_csv("./processed_output.csv")
independent_variables = dataset.drop(columns=["price"])
dependent_variable = dataset["price"]
seed=42

In [ ]:
dataset.info()

In [ ]:
# Scale the data
features_to_scale = ["wheelbase", "carlength", "carwidth", "carheight", "curbweight", "enginesize", "compressionratio", "horsepower", "peakrpm", "highwaympg", "citympg"]
# To avoid data leakage, we first split the data into train and test sets. I will use 80% train, and 20% test set.
X_train, X_test, y_train, y_test = train_test_split(independent_variables, dependent_variable, random_state=seed, test_size=0.2, shuffle=True)

In [ ]:
# DATA IMPUTATION
# https://medium.com/the-modern-scientist/navigating-the-pitfalls-of-data-leakage-in-imputing-missing-values-351091a3963e

fill_mode(dataset=X_train, feature_name='fueltype')
fill_mode(dataset=X_train, feature_name='carbody')
fill_mode(dataset=X_train, feature_name='enginelocation')
fill_mean(dataset=X_train, feature_name="carlength")
fill_mean(dataset=X_train, feature_name="cylindernumber")
fill_mean(dataset=X_train, feature_name="horsepower")

fill_mode(dataset=X_test, feature_name='fueltype')
fill_mode(dataset=X_test, feature_name='carbody')
fill_mode(dataset=X_test, feature_name='enginelocation')
fill_mean(dataset=X_test, feature_name="carlength")
fill_mean(dataset=X_test, feature_name="cylindernumber")
fill_mean(dataset=X_test, feature_name="horsepower")

# DATA CONVERSION
categorical_features = ["carname", "fueltype", "aspiration", "doornumber", "carbody", "drivewheel", "enginelocation", "color"]
for feature in categorical_features:
    X_train, X_test = one_hot_encode_train_test(X_train, X_test, feature)

In [ ]:
# Scale train
for i in X_train:
    if i in features_to_scale:
        z_score_scale(X_train, i)

# Scale test
for i in X_test:
    if i in features_to_scale:
        z_score_scale(X_test, i)

In [ ]:
X_train.describe()

In [ ]:
X_train.info()

In [ ]:
simple_linear_model = LinearRegression()
simple_linear_model.fit(X_train, y_train)
simple_linear_model_preds = simple_linear_model.predict(X_test)
simple_linear_model_preds_on_train = simple_linear_model.predict(X_train)

print("Train set performance: \n")
mae, mse, rmse, r2 = calculate_metrics(y_test=y_train, preds=simple_linear_model_preds_on_train)
print("\nTest set performance: \n")
calculate_metrics(y_test=y_test, preds=simple_linear_model_preds)


In [ ]:
plot_pred_vs_true(y_test, simple_linear_model_preds)

In [ ]:
plot_residual(y_test, simple_linear_model_preds)